In [ ]:
# -----------------------------
# Imports
# -----------------------------
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz


In [ ]:
# -----------------------------
# Gain curve settings
# -----------------------------
# This notebook lives in CODE. The data lives in the repo-level DATA folder,
# so use paths relative to the notebook's parent project folder.
project_folder = Path.cwd().parent if Path.cwd().name == "CODE" else Path.cwd()
data_folder = project_folder / "DATA"
dark_rates_folder = data_folder / "Dark_Rates"

voltage_values = np.array([300, 400, 500, 600, 700, 800, 900])
voltage_folders = [dark_rates_folder / str(V) for V in voltage_values]

R_TERMINATION = 50.0          # Oscilliscope termination, measured in Ohms
T_PRE_SIGNAL = 20e-9          # seconds, pre-pulse region used to measure baseline and noise
PULSE_START_TIME = 20e-9      # seconds, ignore earlier samples when searching for the PMT pulse
THRESHOLD_SIGMA = 5.0         # pulse-height cut, keeps pulses larger than 5 times the noise
APPLY_FILTER = True           # turn the FFT low-pass filter on or off for all waveforms
F_CUTOFF = 1e9                # Hz, cutoff frequency used only if APPLY_FILTER is True
QC_PERCENTILE = 5             # charge cut, removes the smallest charge pulses after the height cut
E_CHARGE = 1.602176634e-19    # Coulombs, charge of one electron used to convert Q_spe into gain
SPE_HIST_BINS = 80            # number of charge-histogram bins used to find the SPE peak


In [ ]:
# -----------------------------
# Utilities copied from PMT_PROCESSING
# -----------------------------
def processing_files(filepath):
    data = np.loadtxt(filepath, delimiter=",", usecols=(0, 1), skiprows=504)
    times = data[:, 0]
    amps = data[:, 1]

    waveforms = []
    current_t = []
    current_a = []

    for i in range(len(times)):
        if i > 0 and times[i] < times[i - 1]:
            waveforms.append((np.array(current_t), np.array(current_a)))
            current_t = []
            current_a = []

        current_t.append(times[i])
        current_a.append(amps[i])

    if current_t:
        waveforms.append((np.array(current_t), np.array(current_a)))

    return waveforms


def normalize_waveform_times(waveforms):
    normed = []
    for t, v in waveforms:
        normed.append((t - t[0], v))
    return normed


def subtract_baseline(waveforms, t_pre_signal=20e-9):
    corrected = []
    baselines = []
    sigmas = []

    for t, v in waveforms:
        mask = t < t_pre_signal

        if np.sum(mask) < 2:
            mask = np.arange(min(10, len(t)))

        baseline = np.mean(v[mask])
        sigma = np.std(v[mask], ddof=1) if np.sum(mask) > 1 else np.std(v[mask])

        corrected.append((t, v - baseline))
        baselines.append(baseline)
        sigmas.append(sigma)

    return corrected, np.array(baselines), np.array(sigmas)


def lowpass_filter_waveform(t, v, f_cut):
    if len(t) < 2:
        return v.copy()

    dt = np.mean(np.diff(t))
    freqs = np.fft.rfftfreq(len(v), d=dt)
    fft_vals = np.fft.rfft(v)

    fft_vals[freqs > f_cut] = 0.0
    v_filt = np.fft.irfft(fft_vals, n=len(v))
    return v_filt


def filter_waveforms(waveforms, f_cut=1e9):
    filtered = []
    for t, v in waveforms:
        filtered.append((t, lowpass_filter_waveform(t, v, f_cut)))
    return filtered


def extract_observables(waveforms, sigmas, pulse_start_time=20e-9, R=50.0):
    H_list = []
    Q_list = []
    sigma_list = []

    for (t, v), sigma in zip(waveforms, sigmas):
        H = -np.min(v)

        mask = t > pulse_start_time
        if np.sum(mask) < 2:
            Q = np.nan
        else:
            Q = (1.0 / R) * np.trapezoid(-v[mask], t[mask])

        H_list.append(H)
        Q_list.append(Q)
        sigma_list.append(sigma)

    return np.array(H_list), np.array(Q_list), np.array(sigma_list)


def waveform_duration(waveforms):
    durations = []
    for t, _ in waveforms:
        if len(t) > 1:
            durations.append(t[-1] - t[0])
    return np.array(durations)


In [ ]:
# -----------------------------
# Pulse-region integration copied from PMT_PROCESSING
# -----------------------------
def integrate_pulse_region(t, v, R=50.0, pulse_start_time=20e-9):
    if len(t) < 3:
        return np.nan, np.nan, np.nan

    search_mask = t >= pulse_start_time
    if np.sum(search_mask) < 3:
        search_mask = np.ones_like(t, dtype=bool)

    idx_candidates = np.where(search_mask)[0]
    idx_min_local = np.argmin(v[search_mask])
    idx_min = idx_candidates[idx_min_local]

    # Skip waveforms that do not contain a negative-going pulse
    if v[idx_min] >= 0:
        return 0.0, t[idx_min], t[idx_min]

    i_left = idx_min
    while i_left > 0 and v[i_left] < 0:
        i_left -= 1

    i_right = idx_min
    while i_right < len(v) - 1 and v[i_right] < 0:
        i_right += 1

    if i_right <= i_left:
        return np.nan, np.nan, np.nan

    q = (1.0 / R) * np.trapezoid(-v[i_left:i_right+1], t[i_left:i_right+1])
    return q, t[i_left], t[i_right]


In [ ]:
# -----------------------------
# Estimate the single-photoelectron charge peak
# -----------------------------
def estimate_spe_charge(charges, bins=80):
    # Keep only real positive charges, since gain must come from positive integrated pulse area
    # This avoids NaNs, zeros, and failed integrations from changing the histogram peak
    charges = np.array(charges)
    charges = charges[np.isfinite(charges) & (charges > 0)]

    # If there are too few pulses, the histogram peak will not mean anything useful
    # Return NaN values so this voltage does not get plotted as a fake gain point
    if len(charges) < 10:
        return np.nan, np.nan, 0

    # Trim off the very lowest and highest charges before looking for the peak
    # This keeps noise tails and large multi-photoelectron pulses from setting the scale
    q_low = np.nanpercentile(charges, 1)
    q_high = np.nanpercentile(charges, 95)
    charges_trim = charges[(charges >= q_low) & (charges <= q_high)]

    # Check again after trimming, because a bad folder can still leave too few pulses
    # Also reject cases where the percentile range collapses to zero width
    if len(charges_trim) < 10 or q_high <= q_low:
        return np.nan, np.nan, 0

    # Build a charge histogram and find the most populated charge bin
    # This is our simple estimate of the single-photoelectron charge peak
    counts, edges = np.histogram(charges_trim, bins=bins)
    idx_peak = np.argmax(counts)
    q_left = edges[max(idx_peak - 1, 0)]
    q_right = edges[min(idx_peak + 2, len(edges) - 1)]

    # Average charges near the peak instead of using just one bin center
    # The standard error of those charges becomes the gain error bar
    peak_charges = charges_trim[(charges_trim >= q_left) & (charges_trim <= q_right)]
    q_spe = np.nanmean(peak_charges) if len(peak_charges) else np.nan
    q_spe_err = np.nanstd(peak_charges, ddof=1) / np.sqrt(len(peak_charges)) if len(peak_charges) > 1 else np.nan

    return q_spe, q_spe_err, len(peak_charges)


# -----------------------------
# Process one voltage folder
# -----------------------------
def process_voltage_folder(data_folder, voltage):
    # Each voltage folder contains many oscilloscope text files
    # This list grabs only the channel-1 waveform files used for the PMT signal
    all_waveforms = []
    target_files = sorted(Path(data_folder).glob("C1C*.txt"))

    # Read every file in the voltage folder and append all waveforms together
    # The processing_files function is reused directly from the PMT processing notebook
    for filepath in target_files:
        file_waveforms = processing_files(filepath)
        all_waveforms.extend(file_waveforms)

    # If a folder is empty or missing usable files, return NaNs for this voltage
    # This keeps the rest of the gain-curve loop from crashing
    if len(all_waveforms) == 0:
        return {
            "Voltage [V]": voltage,
            "Number of files": 0,
            "Number of waveforms": 0,
            "Mean gain": np.nan,
            "Gain error": np.nan,
            "SPE charge [C]": np.nan,
            "SPE charge error [C]": np.nan,
            "SPE peak pulses": 0,
            "Mean accepted charge [C]": np.nan,
            "Accepted pulses N_acc": 0,
            "Accepted pulse rate [Hz]": np.nan,
        }

    # Set every waveform time axis to start at zero
    # This makes the baseline and integration windows line up file to file
    waveforms_norm = normalize_waveform_times(all_waveforms)

    # Measure the baseline before the pulse and subtract it from each waveform
    # The sigma values are the baseline noise and are used for the height threshold
    waveforms_bs, baselines, sigmas = subtract_baseline(
        waveforms_norm,
        t_pre_signal=T_PRE_SIGNAL
    )

    # Optionally low-pass filter the baseline-subtracted waveform
    # This uses the same FFT filter switch and cutoff as the PMT processing notebook
    if APPLY_FILTER:
        waveforms_proc = filter_waveforms(waveforms_bs, f_cut=F_CUTOFF)
    else:
        waveforms_proc = waveforms_bs.copy()

    # Extract a pulse height and rough charge from each waveform
    # The height is used first because it is a simple way to reject noise pulses
    H, Q, sigma_evt = extract_observables(
        waveforms_proc,
        sigmas,
        pulse_start_time=PULSE_START_TIME,
        R=R_TERMINATION
    )

    # Recompute charge using only the negative pulse region around the pulse minimum
    # This avoids integrating a long flat region where baseline noise can add extra area
    Q_region = []
    for t, v in waveforms_proc:
        q_i, tl_i, tr_i = integrate_pulse_region(
            t, v, R=R_TERMINATION, pulse_start_time=PULSE_START_TIME
        )
        Q_region.append(q_i)

    # Convert the charge list to an array and build the pulse-height threshold
    # Only pulses with H greater than THRESHOLD_SIGMA times the baseline noise pass
    Q_region = np.array(Q_region)
    sigma_mean = np.nanmean(sigma_evt)
    V_thr = THRESHOLD_SIGMA * sigma_mean
    height_pass = np.isfinite(H) & (H > V_thr)

    # Use the height-selected charges to define a small charge threshold Qc
    # Qc removes tiny integrals that passed height but do not look like real pulses
    accepted_charge_from_height = Q_region[height_pass & np.isfinite(Q_region)]
    Qc = np.nanpercentile(accepted_charge_from_height, QC_PERCENTILE) if len(accepted_charge_from_height) else np.nan
    charge_pass = np.isfinite(Q_region) & (Q_region >= Qc) if np.isfinite(Qc) else np.zeros(len(Q_region), dtype=bool)

    # Estimate the total live time for this voltage folder
    # This is only used for the accepted pulse rate diagnostic, not for gain
    durations = waveform_duration(waveforms_proc)
    T_window = np.nanmean(durations)
    T_total = len(waveforms_proc) * T_window

    # Q_acc contains the charges that passed the height and charge cuts
    # The mean of Q_acc is printed only as a diagnostic, not used as gain
    Q_acc = Q_region[charge_pass]
    N_acc = len(Q_acc)
    Q_mean_acc = np.nanmean(Q_acc) if N_acc else np.nan
    Q_std_acc = np.nanstd(Q_acc, ddof=1) if N_acc > 1 else np.nan
    Q_err_acc = Q_std_acc / np.sqrt(N_acc) if N_acc > 1 else np.nan
    R_acc = N_acc / T_total if T_total > 0 else np.nan

    # Estimate the single-photoelectron charge from the peak of the charge histogram
    # PMT gain is the number of anode electrons per one photoelectron: gain = Q_spe / e
    Q_spe, Q_spe_err, N_spe = estimate_spe_charge(Q_acc, bins=SPE_HIST_BINS)
    gain_mean = Q_spe / E_CHARGE if np.isfinite(Q_spe) else np.nan
    gain_err = Q_spe_err / E_CHARGE if np.isfinite(Q_spe_err) else np.nan

    return {
        "Voltage [V]": voltage,
        "Number of files": len(target_files),
        "Number of waveforms": len(waveforms_proc),
        "Mean noise sigma [V]": sigma_mean,
        "Voltage threshold V_thr [V]": V_thr,
        "Charge threshold Qc [C]": Qc,
        "Accepted pulses N_acc": int(N_acc),
        "Accepted pulse rate [Hz]": R_acc,
        "SPE charge [C]": Q_spe,
        "SPE charge error [C]": Q_spe_err,
        "SPE peak pulses": int(N_spe),
        "Mean accepted charge [C]": Q_mean_acc,
        "Charge error [C]": Q_err_acc,
        "Mean gain": gain_mean,
        "Gain error": gain_err,
    }


In [ ]:
# -----------------------------
# Run over all voltage folders
# -----------------------------
# This list will hold one dictionary of results for each voltage folder
# Each dictionary contains counts, charges, and the final SPE gain for that voltage
gain_results = []

# Loop over the voltage labels and their matching folders together
# process_voltage_folder does the full waveform processing for one voltage at a time
for voltage, folder in zip(voltage_values, voltage_folders):
    print(f"Processing {voltage} V: {folder}")
    result = process_voltage_folder(folder, voltage)
    gain_results.append(result)

print("Done")


In [ ]:
# -----------------------------
# Print gain table
# -----------------------------
# Print the main numbers so you can check the calculation before trusting the plot
# Mean accepted charge is diagnostic, while SPE gain is the value used for the gain curve
print("========== GAIN CURVE RESULTS ==========")
for result in gain_results:
    print(f"{result['Voltage [V]']:4.0f} V")
    print(f"  Files                 : {result['Number of files']}")
    print(f"  Waveforms             : {result['Number of waveforms']}")
    print(f"  Accepted pulses       : {result['Accepted pulses N_acc']}")
    print(f"  Mean accepted charge  : {result['Mean accepted charge [C]']:.4e} C")
    print(f"  SPE charge            : {result['SPE charge [C]']:.4e} C")
    print(f"  SPE peak pulses       : {result['SPE peak pulses']}")
    print(f"  SPE gain              : {result['Mean gain']:.4e}")
    print(f"  Gain error            : {result['Gain error']:.4e}")


In [ ]:
# -----------------------------
# Make arrays for plotting
# -----------------------------
# Pull the voltage, gain, gain error, and diagnostic quantities out of gain_results
# Arrays are easier to mask, plot, and compare than a list of dictionaries
V_curve = np.array([r["Voltage [V]"] for r in gain_results])
G_curve = np.array([r["Mean gain"] for r in gain_results])
G_err = np.array([r["Gain error"] for r in gain_results])
Q_curve = np.array([r["SPE charge [C]"] for r in gain_results])
N_acc_curve = np.array([r["Accepted pulses N_acc"] for r in gain_results])

# Only plot points with finite positive gain values
# Log-scale plots cannot show zero, negative, or NaN gain values
valid_gain = np.isfinite(V_curve) & np.isfinite(G_curve) & (G_curve > 0)

# Print the exact arrays that go into the gain curve
# This makes it easier to catch one bad voltage point before looking at the graph
print(f"Voltages in gain curve = {V_curve[valid_gain]}")
print(f"Gains in gain curve    = {G_curve[valid_gain]}")


In [ ]:
# -----------------------------
# Load listed R7378A gain curve from JSON
# -----------------------------
# The JSON came from a log-y plot digitization. Its Gain dataset values are stored
# on the digitized axis scale, so LISTED_GAIN_SCALE converts them onto ordinary PMT gain.
listed_gain_json = data_folder / "R7378A_Characteristics.json"
LISTED_GAIN_SCALE = 1e13

with listed_gain_json.open("r") as f:
    listed_gain_data = json.load(f)

listed_gain_points = []
for dataset in listed_gain_data.get("datasetColl", []):
    if dataset.get("name") == "Gain":
        for point in dataset.get("data", []):
            if "value" in point and len(point["value"]) >= 2:
                listed_gain_points.append(point["value"][:2])

listed_gain_points = np.array(listed_gain_points, dtype=float)
order = np.argsort(listed_gain_points[:, 0])
V_listed = listed_gain_points[order, 0]
G_listed_raw = listed_gain_points[order, 1]
G_listed = G_listed_raw * LISTED_GAIN_SCALE

# Interpolate/extrapolate in log(gain), because PMT gain curves are approximately log-linear.
valid_listed = np.isfinite(V_listed) & np.isfinite(G_listed) & (G_listed > 0)
V_listed = V_listed[valid_listed]
G_listed = G_listed[valid_listed]
logG_listed = np.log10(G_listed)

# Fit log10(gain) vs voltage so the listed curve can be extrapolated below/above
# the digitized voltage range. This keeps the existing interpolation but avoids
# flat extrapolation outside the JSON points.
listed_fit_coeff = np.polyfit(V_listed, logG_listed, deg=1)

def interp_extrap_log_gain(V_query, V_data, logG_data, fit_coeff):
    V_query = np.array(V_query, dtype=float)
    logG_query = np.interp(V_query, V_data, logG_data)

    below = V_query < np.nanmin(V_data)
    above = V_query > np.nanmax(V_data)
    outside = below | above
    if np.any(outside):
        logG_query[outside] = np.polyval(fit_coeff, V_query[outside])

    return 10 ** logG_query

V_min_plot = np.nanmin([np.nanmin(V_curve[valid_gain]), np.nanmin(V_listed)])
V_max_plot = np.nanmax([np.nanmax(V_curve[valid_gain]), np.nanmax(V_listed)])
V_listed_interp = np.linspace(V_min_plot, V_max_plot, 500)
G_listed_interp = interp_extrap_log_gain(V_listed_interp, V_listed, logG_listed, listed_fit_coeff)
G_listed_at_measured = interp_extrap_log_gain(V_curve, V_listed, logG_listed, listed_fit_coeff)

# Smooth experimental curve through the measured points. This is interpolation only
# over the measured voltage range, not an extrapolated model.
valid_exp_interp = valid_gain & np.isfinite(G_curve) & (G_curve > 0)
V_exp = V_curve[valid_exp_interp]
G_exp = G_curve[valid_exp_interp]
exp_order = np.argsort(V_exp)
V_exp = V_exp[exp_order]
G_exp = G_exp[exp_order]
V_exp_interp = np.linspace(np.nanmin(V_exp), np.nanmax(V_exp), 400)
G_exp_interp = 10 ** np.interp(V_exp_interp, V_exp, np.log10(G_exp))

print(f"Loaded listed gain curve from {listed_gain_json}")
print(f"Listed gain voltages: {V_listed}")
print(f"Listed gain values after scale factor {LISTED_GAIN_SCALE:.1e}: {G_listed}")
print(f"Listed log-space extrapolation fit: log10(G) = {listed_fit_coeff[0]:.4e} V + {listed_fit_coeff[1]:.4e}")
print(f"Listed gain interpolated/extrapolated at measured voltages: {G_listed_at_measured[valid_gain]}")


In [ ]:
# -----------------------------
# Plot measured gain curve over listed R7378A curve
# -----------------------------
# The listed curve is interpolated between JSON points and extrapolated outside
# the JSON voltage range using the fitted log-space trend.
# The experimental curve is log-space interpolation through measured points only.
plt.figure(figsize=(9, 5.5))

plt.plot(
    V_listed_interp,
    G_listed_interp,
    color="tab:blue",
    ls="-",
    lw=2.5,
)
plt.plot(
    V_listed,
    G_listed,
    "s",
    color="tab:cyan",
    markeredgecolor="navy",
    ms=5,
    label="R7378A",
)
plt.plot(
    V_exp_interp,
    G_exp_interp,
    color="tab:red",
    ls="--",
    lw=2.5,
)
plt.errorbar(
    V_curve[valid_gain],
    G_curve[valid_gain],
    yerr=G_err[valid_gain],
    fmt="o",
    color="black",
    ecolor="tab:orange",
    markerfacecolor="gold",
    markeredgecolor="black",
    markersize=6,
    elinewidth=1.5,
    capsize=4,
    label="Measured gain",
)

plt.yscale("log")
plt.xlabel("PMT Voltage [V]")
plt.xlim(300,900)
plt.ylabel("Gain")
plt.ylim(1e3,1e7)
plt.title("Measured PMT Gain vs Listed R7378A Gain")
plt.legend()
plt.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()
